# CryptoMind 內容檢查：微調 Laya（過／不過）

**開始前，右側 Settings 設定：**
- Accelerator：**GPU T4 x2**（只會用到一張）
- Internet：**On**（要下載官方 Laya）
- Input → **Add Input** → 加入你上傳的內容檢查資料集（v2 起上傳成同一個資料集的新版本）

**跑法：**
- 想先確認環境：把下面的 `SMOKE` 改成 `True`，跑完幾分鐘沒報錯，再改回 `False`
- 正式訓練：右上 **Save Version → Save & Run All (Commit)**，可以關掉瀏覽器，跑完到 Output 下載 `laya-moderation.zip`

In [ ]:
# 1. 找到上傳的資料、確認有 GPU
import glob
import os

import torch

hits = glob.glob('/kaggle/input/**/train.py', recursive=True)
assert hits, '找不到 train.py：右側 Input → Add Input 加入上傳的內容檢查資料集'
DATA = os.path.dirname(hits[0])
print('資料：', DATA, sorted(os.listdir(DATA)))
assert torch.cuda.is_available(), '沒有 GPU：右側 Settings → Accelerator 選 GPU T4 x2'
print('GPU：', torch.cuda.get_device_name(0))

In [ ]:
# 2. 安裝套件（Laya、LoRA、ONNX 匯出）
# Kaggle 預裝的 torchao 0.10 太舊，新版 peft 偵測到就直接報錯（沒裝則略過）；我們用不到，先移除
!pip uninstall -y -q torchao
!pip install -q laya peft onnx onnxruntime

In [ ]:
# 3. 訓練（正式約 30～60 分鐘；SMOKE=True 只跑少量確認流程，幾分鐘）
SMOKE = False
OUT = '/kaggle/working/laya-moderation'
!python {DATA}/train.py --data {DATA} --out {OUT} {'--smoke' if SMOKE else ''}

In [ ]:
# 4. 成績摘要（完整明細在上一格的輸出）
import json

m = json.load(open(f'{OUT}/metrics.json', encoding='utf-8'))
before, after = m['before_finetune'], m['test']
print(f"微調前：不過擋下 {before['block_rate']:.1%}，過被誤擋 {before['false_block_rate']:.1%}")
print(f"微調後：不過擋下 {after['block_rate']:.1%}，過被誤擋 {after['false_block_rate']:.1%}（第 {m['best_epoch']} 輪，門檻 {after['threshold']:.4f}）")
print('達標 ✓' if m['passed'] else '還沒達標 ✗（看上一格「達標線」哪幾條沒過）')

In [ ]:
# 5. 下載：點下面的連結，或右側 Output 找 laya-moderation.zip（約 600 MB），拿回來給 Claude
from IPython.display import FileLink

os.chdir('/kaggle/working')
FileLink('laya-moderation.zip')